<a href="https://colab.research.google.com/github/drizzy765/flyrank_internship_ML-01_assignment/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Week 1: Research Question Framing

**This notebook defines the business logic and risk profile for my FlyRank capstone project, alongside a brief data exploration to validate the chosen lane**

## 1) My Lane and Why

Chosen Lane: CTR / Engagement Opportunity Scoring (Core Lane)

Why: My background as a Quantitative Researcher involves identifying deviations and volatility in time-series datasets (e.g., Nigerian commodity pricing). Detecting URLs that significantly underperform their expected Click-Through Rate (CTR) based on their ranking position is mathematically similar to identifying pricing anomalies. It requires multivariate modeling to establish a baseline expectation, which perfectly aligns with my skill set in regression and anomaly detection.

## 2.) The Question: Decision, Action, Cost of a Wrong Call

The Search Question: Which highly visible, high-ranking pages are severely under-capturing clicks (CTR anomaly) based on their current search position and historical engagement profile?

Unit of Analysis: A specific, pseudonymized content item (content_hash_id) aggregated over a 90-day window.

The Output: A ranked queue of review candidates, featuring a predicted "expected CTR" versus "actual CTR" delta, alongside reason codes (e.g., strong position, high impressions, low CTR).

The Decision & Action: A Content Manager or SEO Specialist will review the top 50 flagged URLs on this list. For valid anomalies, they will rewrite the meta-titles, descriptions, or adjust the on-page snippet structure to recapture lost clicks.

Cost of a Wrong Call (Risk Profile):

False Positive (Flagging a healthy URL): The SEO team wastes hours tweaking a page that is already performing optimally, potentially causing Google to drop its rank due to unnecessary changes.

False Negative (Missing a broken URL): The business continues to bleed high-intent organic traffic, losing potential revenue to competitors ranking lower but earning more clicks.

Why ML / Data? Expected CTR is not a static curve; it fluctuates based on query intent, device type, age, and position. Simple SQL averages cannot account for these non-linear, multivariate interactions. A learned ranking model will outperform static rules.

## 3) Quick Look at the Data

The code below loads the starter dataset to prove there is enough variance and volume to support building a CTR expectation model

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import os

In [ ]:
data_path = 'content_refresh_anonymized.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path)

    print("--- Data Sanity Check ---")
    print(f"Total rows in starter dataset: {len(df):,}")

    # Check 1: Do we have pages with high visibility but low CTR?
    # Filtering for pages ranking on Page 1 (pos 1-10) with significant impressions
    high_vis_pages = df[(df['avg_position'] <= 10) & (df['impressions_90d'] > 1000)]

    if not high_vis_pages.empty:
        avg_ctr_page_one = high_vis_pages['ctr'].mean()
        print(f"\nAverage CTR for Page 1 content (>1k impressions): {avg_ctr_page_one:.2%}")

        # Identify anomalies: Page 1 rank, >1k impressions, but < 1% CTR
        severe_anomalies = high_vis_pages[high_vis_pages['ctr'] < 0.01]
        print(f"Number of severe anomalies (Page 1, High Impressions, <1% CTR): {len(severe_anomalies):,}")

        print("\nConclusion: The presence of severe anomalies indicates a strong target for predictive CTR scoring.")
    else:
        print("\nCould not calculate baseline metrics; dataset may require different thresholds.")
else:
    print(f"Dataset not found. Please upload '{data_path}' to your Colab session.")

--- Data Sanity Check ---
Total rows in starter dataset: 30,000

Average CTR for Page 1 content (>1k impressions): 34.53%
Number of severe anomalies (Page 1, High Impressions, <1% CTR): 259

Conclusion: The presence of severe anomalies indicates a strong target for predictive CTR scoring.


## 4) Careful Words: What I Can and Can't Claim

What I CAN claim: The system can estimate the expected CTR for a given content item based on its historical performance features and flag instances where performance falls significantly below that statistical expectation. It can rank these opportunities to optimize human review time.

What I CANNOT claim: The system cannot guarantee that rewriting a meta-title will immediately fix the CTR, nor can it definitively prove why the anomaly occurred (e.g., a competitor launching a better snippet, or a shift in Google's SERP layout). I cannot claim that a "refresh caused recovery" without a valid causal design experiment. The model highlights probabilistic symptoms, not definitive causes.





## 5) Self-Check

[x] Picked one of the four predefined lanes (CTR / Engagement Opportunity Scoring).

[x] Named the decision (rewrite meta/snippet) and the action (Content Manager review).

[x] Showed at least two real numbers from the starter data in the code cell above.

[x] Explained why this is not just "train a model" (translating expected CTR delta into a prioritized human queue).

[x] Used careful language regarding causal claims.